# Cooling Tower Inspections: Bronx vs. Manhattan

**Thesis:** Both boroughs went from near-universal violations (2017) to low violation rates, but since 2019 the Bronx has consistently lagged Manhattan, especially on the most serious violations (Critical and Public Health Hazard).

**Setup:** `pip install pandas plotly folium scipy nbformat`, and put the CSV in a `data/` folder next to this notebook.

## 1. Load and clean the data
The raw file has one row per *violation*, so we collapse it to one row per *inspection* before computing any rates.

In [13]:
%pip install pandas plotly folium scipy nbformat ipykernel

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os, glob
print("Notebook is running in:", os.getcwd())
print("CSV files found:", glob.glob("**/*.csv", recursive=True))

Notebook is running in: /workspaces/2026-Image-and-Sound/lessons/Final_Project
CSV files found: ['NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv']


In [15]:
import glob
CSV = glob.glob("**/*Cooling_Tower*.csv", recursive=True)[0]
print("Using:", CSV)

Using: NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv


In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from scipy.stats import chi2_contingency
import os

CSV = "NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv"
os.makedirs("charts", exist_ok=True)

df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["Inspection_date"])
df["year"] = df["date"].dt.year
df["has_violation"] = df["Violation_Code"].notna()

# IMPORTANT: the raw file has one row per VIOLATION, so a single inspection can
# appear many times. Collapse to one row per inspection before computing rates.
insp = (
    df.groupby(["System_ID", "Inspection_date", "Borough", "year", "Inspection_Type"])
    .agg(
        any_violation=("has_violation", "max"),
        n_violations=("has_violation", "sum"),
        critical=("Violation_Type", lambda s: (s == "Critical").any()),
        phh=("Violation_Type", lambda s: (s == "PHH").any()),
    )
    .reset_index()
)

COLORS = {"Bronx": "#d62728", "Manhattan": "#1f77b4"}
print(f"{len(df):,} violation rows -> {len(insp):,} inspections")
print(insp.groupby("Borough").size())

95,893 violation rows -> 29,450 inspections
Borough
Bronx         2204
Manhattan    27246
dtype: int64


## 2. Headline trend
Both boroughs improved sharply, but the Bronx has trailed since 2019.

In [17]:
trend = insp.groupby(["year", "Borough"])["any_violation"].mean().reset_index()
trend["pct"] = trend["any_violation"] * 100

fig1 = px.line(
    trend, x="year", y="pct", color="Borough", markers=True,
    color_discrete_map=COLORS,
    labels={"pct": "% of inspections with a violation", "year": "Year"},
    title="Both boroughs improved sharply, but the Bronx has trailed since 2019",
)
fig1.add_vrect(x0=2025.5, x1=2026.5, fillcolor="gray", opacity=0.15,
               annotation_text="2026 partial year", annotation_position="top left")
fig1.update_layout(yaxis_range=[0, 100])
fig1.write_html("charts/1_trend.html")
fig1.show()

## 3. Severity
The gap is widest for the most serious violation types.

In [18]:
sev = insp.groupby(["year", "Borough"])[["critical", "phh"]].mean().mul(100).reset_index()
sev = sev.melt(id_vars=["year", "Borough"], var_name="Severity", value_name="pct")
sev["Severity"] = sev["Severity"].map({"critical": "Critical violation", "phh": "Public health hazard"})

fig2 = px.line(
    sev[sev["year"] >= 2019], x="year", y="pct", color="Borough",
    facet_col="Severity", markers=True, color_discrete_map=COLORS,
    labels={"pct": "% of inspections", "year": "Year"},
    title="The gap is widest for the most serious violations (2019 onward)",
)
fig2.write_html("charts/2_severity.html")
fig2.show()

## 4. Sample-size context
Manhattan has far more towers, so Bronx rates rest on smaller samples.

In [19]:
vol = insp.groupby(["year", "Borough"]).size().reset_index(name="inspections")
fig3 = px.bar(
    vol, x="year", y="inspections", color="Borough", barmode="group",
    color_discrete_map=COLORS,
    title="Inspection volume: Manhattan has far more towers, so Bronx rates rest on smaller samples",
)
fig3.write_html("charts/3_volume.html")
fig3.show()

## 5. Violation mix
Are the boroughs failing in different ways, or the same ways?

In [20]:
viol = df[df["has_violation"]].copy()
viol["short"] = viol["Violation_Text"].str.slice(0, 60)
top_codes = viol["Violation_Code"].value_counts().head(8).index
mix = (
    viol[viol["Violation_Code"].isin(top_codes)]
    .groupby("Borough")["short"].value_counts(normalize=True)
    .mul(100).rename("pct").reset_index()
)
fig4 = px.bar(
    mix, y="short", x="pct", color="Borough", barmode="group", orientation="h",
    color_discrete_map=COLORS,
    labels={"pct": "% of that borough's violations", "short": ""},
    title="Same failures in both boroughs: the Bronx problem is frequency, not type",
)
fig4.update_layout(height=550, yaxis={"categoryorder": "total ascending"})
fig4.write_html("charts/4_mix.html")
fig4.show()

## 6. Statistical check
Chi-square tests on the 2022-2025 window.

In [21]:
recent = insp[insp["year"].between(2022, 2025)]
for col in ["any_violation", "critical", "phh"]:
    ct = pd.crosstab(recent["Borough"], recent[col])
    p = chi2_contingency(ct)[1]
    rates = recent.groupby("Borough")[col].mean().mul(100).round(1).to_dict()
    print(f"{col:14s} {rates}  chi-square p = {p:.2g}")

any_violation  {'Bronx': 19.4, 'Manhattan': 14.0}  chi-square p = 9.5e-06
critical       {'Bronx': 15.9, 'Manhattan': 10.7}  chi-square p = 1.1e-06
phh            {'Bronx': 11.2, 'Manhattan': 6.9}  chi-square p = 1.7e-06


## 7. Map of towers
One dot per cooling tower system, colored by violation rate since 2022.

In [24]:
# One dot per cooling tower SYSTEM, using inspections from 2022 onward.
recent_sys = (
    insp[insp["year"] >= 2022]
    .groupby("System_ID")
    .agg(rate=("any_violation", "mean"), n=("any_violation", "size"))
    .reset_index()
)
loc = df.drop_duplicates("System_ID")[
    ["System_ID", "Address", "Borough", "Latitude", "Longitude", "Status"]
].dropna(subset=["Latitude", "Longitude"])
sysmap = recent_sys.merge(loc, on="System_ID")

m = folium.Map(location=[40.80, -73.92], zoom_start=11, tiles="OpenStreetMap")

def color(rate):
    return "#2ca02c" if rate == 0 else "#ff7f0e" if rate < 0.5 else "#d62728"

for _, r in sysmap.iterrows():
    folium.CircleMarker(
        [r["Latitude"], r["Longitude"]], radius=4,
        color=color(r["rate"]), fill=True, fill_opacity=0.7, weight=1,
        tooltip=f"{r['Address']} ({r['Borough']}): {r['rate']:.0%} of {r['n']} inspections had violations",
    ).add_to(m)

legend = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
padding:8px 12px;border:1px solid #999;font-size:13px">
<b>Violations, 2022+</b><br>
<span style="color:#2ca02c">&#9679;</span> none<br>
<span style="color:#ff7f0e">&#9679;</span> under half of inspections<br>
<span style="color:#d62728">&#9679;</span> half or more</div>"""
m.get_root().html.add_child(folium.Element(legend))
m.save("charts/5_tower_map.html")
m   # displays in a notebook; otherwise open charts/5_tower_map.html

## 8. Neighborhoods
Highest-violation NTAs (minimum 60 inspections).

In [23]:
nta_lookup = df.drop_duplicates("System_ID")[["System_ID", "NTA_Code"]]
nta = (
    insp[insp["year"] >= 2019]
    .merge(nta_lookup, on="System_ID", how="left")
    .groupby(["Borough", "NTA_Code"])
    .agg(rate=("any_violation", "mean"), inspections=("any_violation", "size"))
    .reset_index()
)
nta = nta[nta["inspections"] >= 60]          # skip tiny samples
nta["pct"] = nta["rate"] * 100
top = nta.sort_values("pct", ascending=False).head(15)

fig6 = px.bar(
    top, x="pct", y="NTA_Code", color="Borough", orientation="h",
    color_discrete_map=COLORS, hover_data=["inspections"],
    labels={"pct": "% of inspections with a violation (2019+)", "NTA_Code": "Neighborhood (NTA code)"},
    title="Highest-violation neighborhoods (minimum 60 inspections)",
)
fig6.update_layout(yaxis={"categoryorder": "total ascending"})
fig6.write_html("charts/6_nta.html")
fig6.show()
# Tip: join NTA_Code to the NYC NTA shapefile/GeoJSON from NYC Open Data
# (or look the codes up) to turn this into a choropleth and get neighborhood names.